# Kaggle CPU Server (Accelerator: None — 30GB RAM / 50GB+ SSD): Chạy Full Giờ Bonsai 2 27B Uncensored + Vision (Nhìn Ảnh) từ Dataset Note 1

Notebook này được gộp gọn trong **1 file duy nhất** để chạy trực tiếp bộ **Dataset đã tạo từ Notebook 1** trên **Kaggle CPU (`Accelerator: None`)** cho đến khi chạm giới hạn thời gian phiên (tối đa **12 giờ liên tục/phiên**, **0 tốn quota GPU**).

### Chuẩn bị trước khi bấm `Run All` trên Kaggle:
1. **Gắn Dataset từ Note 1**: Mở bảng **Input** bên phải -> **Add Input** -> **Your Datasets** -> chọn Dataset `bonsai_bundle` bạn đã lưu ở Note 1 (chứa sẵn 2 model: **Model AI 27B** `7.66 GB` và **Model nhìn ảnh Vision mmproj** `0.63 GB`).
2. **Chỉnh Accelerator về CPU**: Trong **Session options** -> **Accelerator** -> chọn **None (CPU - 30GB RAM, ~50GB+ SSD)**.
3. **Bật Internet**: Chuyển **Internet** sang **ON** (để mở Ngrok kết nối với RikkaHub / WebUI).
4. **Bật Secrets**: Vào **Add-ons -> Secrets**, tick chọn `NGROK_AUTHTOKEN` (và `NGROK_DOMAIN` nếu dùng tên miền tĩnh).

## 1. Cấu hình Server CPU (30GB RAM / 50GB SSD), 2 Model (AI + Vision) & Keep-Alive Full Giờ

In [ ]:
import glob
import json
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
import urllib.request

# Tự động nhận diện nền tảng
if os.path.exists("/kaggle/working"):
    PLATFORM = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content"):
    PLATFORM = "colab"
else:
    PLATFORM = "local"

# ==========================================
# 1. CẤU HÌNH 2 MODEL (AI CHÍNH + MODEL NHÌN ẢNH VISION)
# ==========================================
REPO_ID = "BoldingBuilds/Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP-GGUF"
PRISM_RELEASE_TAG = "prism-b10828-a153697"

# Model 1 (AI chính - lấy từ Dataset Note 1):
PREFER_MTP_FILE = True          # True: ưu tiên tìm file *-PQ2_0-MTP.gguf (7.66 GB) đã tải ở Note 1
USE_MTP_DRAFTING = False        # Trên CPU 4 vCPU nên để False để tránh nghẽn CPU và tránh lỗi khi gửi ảnh Vision

# Model 2 (Model nhìn ảnh Vision - lấy từ Dataset Note 1):
USE_VISION = True               # True: nạp kèm Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf (0.63 GB)
IMAGE_MIN_TOKENS = 512          # Số token tối thiểu cho mỗi ảnh trên CPU
IMAGE_MAX_TOKENS = 1024         # Giới hạn token ảnh (1024 giúp CPU xử lý ảnh nhanh hơn mà vẫn đọc rõ chữ)

# ==========================================
# 2. TỐI ƯU HÓA CHO KAGGLE CPU (30GB RAM & 50GB+ SSD)
# ==========================================
CPU_THREADS = os.cpu_count() or 4  # Tự động dùng toàn bộ luồng CPU của Kaggle (4 vCPU)
CONTEXT_SIZE = 32768               # 32K context mặc định trên CPU (có thể nâng lên 65536 hoặc 131072 trên 30GB RAM)
KV_CACHE_TYPE = "q8_0"             # Nén KV cache 8-bit: tiết kiệm ~50% RAM
COPY_TO_LOCAL_SSD = True           # QUAN TRỌNG: Copy 2 model từ /kaggle/input (FUSE chậm) sang SSD cục bộ (/tmp) + Page Cache 30GB RAM
LOAD_MODE = "mmap"                 # Dùng 'mmap' trên SSD cục bộ để khởi động siêu nhanh, không bị treo I/O
BATCH_SIZE = 512
UBATCH_SIZE = 512

DEFAULT_THINKING = False           # False = tắt thinking mặc định để phản hồi nhanh trên CPU; True = bật thinking
REASONING_EFFORT = "low"           # Nếu bật thinking trên CPU nên chọn 'low' hoặc 'medium'
SERVER_PORT = 8081

# Điền tay nếu không dùng Kaggle Secrets:
MANUAL_NGROK_AUTHTOKEN = ""
MANUAL_NGROK_DOMAIN = ""

# ==========================================
# 3. CẤU HÌNH KEEP-ALIVE CHẠY FULL GIỜ KAGGLE (TỐI ĐA 12 GIỜ/PHIÊN)
# ==========================================
ENABLE_KEEP_ALIVE = True            # True: giữ server chạy liên tục cho đến khi chạm mức giờ Kaggle
KEEP_ALIVE_UNTIL_LIMIT = True       # True: chạy không giới hạn cho tới khi Kaggle tự ngắt phiên (12h+); False: dừng theo KEEP_ALIVE_MAX_HOURS
KEEP_ALIVE_MAX_HOURS = 12.0         # Mức giờ tối đa của 1 phiên Kaggle CPU (12.0 giờ)
KEEP_ALIVE_PING_INTERVAL = 45       # Chu kỳ kiểm tra sức khỏe server & ghi heartbeat chống idle (giây)
KEEP_ALIVE_AUTO_RESTART = True      # Tự động bật lại llama-server hoặc Ngrok nếu bị ngắt giữa chừng
KEEP_ALIVE_ENABLE_BROWSER_JS = True # Chèn JS chống ngủ tab trình duyệt

# ==========================================
# KIỂM TRA PHẦN CỨNG CPU / 30GB RAM / 50GB SSD
# ==========================================
def get_system_specs():
    cpu_name = "Unknown CPU"
    try:
        with open("/proc/cpuinfo", "r") as f:
            for line in f:
                if "model name" in line:
                    cpu_name = line.split(":", 1)[1].strip()
                    break
    except Exception:
        pass

    ram_total_gb, ram_avail_gb = 0.0, 0.0
    try:
        with open("/proc/meminfo", "r") as f:
            mem = f.read()
        mt = re.search(r"MemTotal:\s+(\d+)\s+kB", mem)
        ma = re.search(r"MemAvailable:\s+(\d+)\s+kB", mem)
        if mt:
            ram_total_gb = int(mt.group(1)) / (1024 ** 2)
        if ma:
            ram_avail_gb = int(ma.group(1)) / (1024 ** 2)
    except Exception:
        pass

    disk_check_path = "/kaggle/working" if os.path.exists("/kaggle/working") else "/"
    du = shutil.disk_usage(disk_check_path)
    ssd_total_gb = du.total / (1024 ** 3)
    ssd_free_gb = du.free / (1024 ** 3)
    return cpu_name, ram_total_gb, ram_avail_gb, ssd_total_gb, ssd_free_gb

cpu_name, ram_tot, ram_avail, ssd_tot, ssd_free = get_system_specs()
print("=" * 72)
print(f"NỀN TẢNG          : {PLATFORM.upper()} (Chế độ CPU — Không tốn Quota GPU)")
print(f"CPU               : {cpu_name} ({CPU_THREADS} vCPU threads)")
print(f"RAM HỆ THỐNG      : {ram_avail:.1f} GB khả dụng / {ram_tot:.1f} GB tổng")
print(f"SSD LƯU TRỮ       : {ssd_free:.1f} GB trống / {ssd_tot:.1f} GB tổng")
print(f"CHẾ ĐỘ NẠP MODEL  : Copy sang SSD cục bộ + {LOAD_MODE} (tránh treo FUSE /kaggle/input)")
print(f"KEEP-ALIVE        : {'Chạy FULL GIỜ đến khi chạm giới hạn Kaggle (~12h)' if KEEP_ALIVE_UNTIL_LIMIT else f'{KEEP_ALIVE_MAX_HOURS} giờ'}")
print("=" * 72)

## 2. Nạp Binary CPU & Copy 2 Model từ Dataset Note 1 sang SSD Cục Bộ

In [ ]:
!pip install -q pyngrok hf_transfer huggingface_hub requests

import requests
from huggingface_hub import hf_hub_download
from pyngrok import conf, ngrok
from pyngrok.installer import install_ngrok

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SEARCH_ROOTS = []
if PLATFORM == "kaggle":
    SEARCH_ROOTS.extend(["/kaggle/input", "/kaggle/working"])
elif PLATFORM == "colab":
    SEARCH_ROOTS.extend(["/content/drive/MyDrive/bonsai_bundle", "/content/bonsai_bundle", "/content"])
else:
    SEARCH_ROOTS.append(os.getcwd())

LOCAL_SSD_DIR = "/tmp/bonsai_ssd"
os.makedirs(LOCAL_SSD_DIR, exist_ok=True)


def find_existing_file(filename_or_pattern):
    for root_dir in SEARCH_ROOTS:
        if os.path.exists(root_dir):
            matches = glob.glob(os.path.join(root_dir, "**", filename_or_pattern), recursive=True)
            if matches:
                return matches[0]
    return None


def stage_to_local_ssd(src_path, label):
    """Copy file từ /kaggle/input (FUSE chậm) sang SSD cục bộ (/tmp) và hiển thị tốc độ."""
    if not COPY_TO_LOCAL_SSD:
        return src_path
    dst_path = os.path.join(LOCAL_SSD_DIR, os.path.basename(src_path))
    src_size = os.path.getsize(src_path)
    if os.path.exists(dst_path) and os.path.getsize(dst_path) == src_size:
        print(f"✅ [{label}] Đã có sẵn trên SSD cục bộ: {dst_path}")
        return dst_path
    print(f"Đang copy [{label}] ({src_size / (1024**3):.2f} GB) từ Dataset sang SSD cục bộ (vừa nạp vào Page Cache 30GB RAM)...")
    t0 = time.time()
    copied = 0
    last_print = t0
    chunk_size = 16 * 1024 * 1024  # 16 MB buffer
    with open(src_path, "rb") as fsrc, open(dst_path, "wb") as fdst:
        while True:
            buf = fsrc.read(chunk_size)
            if not buf:
                break
            fdst.write(buf)
            copied += len(buf)
            now = time.time()
            if now - last_print >= 3.0:
                pct = copied * 100.0 / src_size
                spd = (copied / (1024**2)) / max(0.1, now - t0)
                print(f"   -> {pct:5.1f}% ({copied / (1024**3):.2f}/{src_size / (1024**3):.2f} GB) | {spd:.0f} MB/s")
                last_print = now
    dt = max(0.1, time.time() - t0)
    print(f"✅ [{label}] Đã copy xong sang SSD cục bộ trong {dt:.1f}s ({(src_size / (1024**2)) / dt:.0f} MB/s)!")
    return dst_path


def fix_permissions_and_symlinks(target_dir):
    for root, _, files in os.walk(target_dir):
        for fn in files:
            fp = os.path.join(root, fn)
            if not os.path.islink(fp):
                os.chmod(fp, 0o755)
            m = re.match(r"^((lib.+\.so)\.0)\..+$", fn)
            if m:
                so0_name, so_name = m.group(1), m.group(2)
                for link_name in (so0_name, so_name):
                    link_path = os.path.join(root, link_name)
                    if not os.path.exists(link_path):
                        os.symlink(fn, link_path)


def test_cpu_binary(bin_path):
    bin_dir = os.path.dirname(bin_path)
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = f"{bin_dir}:" + env.get("LD_LIBRARY_PATH", "")
    res = subprocess.run([bin_path, "--version"], env=env, capture_output=True, text=True)
    return res.returncode == 0, (res.stdout + "\n" + res.stderr).strip()


# ==========================================
# 1. CHUẨN BỊ BINARY LLAMA-SERVER CHO CPU
# ==========================================
TMP_BIN_DIR = "/tmp/prism-cpu-bin"
LLAMA_SERVER_BIN = None

try:
    if os.path.exists(TMP_BIN_DIR):
        shutil.rmtree(TMP_BIN_DIR)
    os.makedirs(TMP_BIN_DIR, exist_ok=True)

    cpu_tar_url = f"https://github.com/PrismML-Eng/llama.cpp/releases/download/{PRISM_RELEASE_TAG}/llama-{PRISM_RELEASE_TAG}-bin-ubuntu-x64.tar.gz"
    tar_path = "/tmp/prism-ubuntu-x64.tar.gz"
    print(f"Đang tải binary thuần CPU (llama-{PRISM_RELEASE_TAG}-bin-ubuntu-x64.tar.gz)...")
    try:
        urllib.request.urlretrieve(cpu_tar_url, tar_path)
    except Exception:
        api_url = f"https://api.github.com/repos/PrismML-Eng/llama.cpp/releases/tags/{PRISM_RELEASE_TAG}"
        req = urllib.request.Request(api_url, headers={"User-Agent": "kaggle-cpu-runner"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            rel_data = json.loads(resp.read().decode("utf-8"))
        asset = next(
            (a for a in rel_data.get("assets", []) if ("ubuntu-x64" in a["name"] or "linux-cpu" in a["name"]) and a["name"].endswith(".tar.gz")),
            None,
        )
        if not asset:
            raise RuntimeError("Không tìm thấy gói ubuntu-x64 trên GitHub Release.")
        urllib.request.urlretrieve(asset["browser_download_url"], tar_path)

    with tarfile.open(tar_path, "r:gz") as tf:
        tf.extractall(TMP_BIN_DIR)
    os.remove(tar_path)
    fix_permissions_and_symlinks(TMP_BIN_DIR)
    found = glob.glob(os.path.join(TMP_BIN_DIR, "**", "llama-server"), recursive=True)
    if found:
        ok, out = test_cpu_binary(found[0])
        if ok:
            LLAMA_SERVER_BIN = found[0]
            print(f"✅ Đã nạp binary thuần CPU thành công: {LLAMA_SERVER_BIN}")
except Exception as e:
    print(f"⚠️ Không tải được gói CPU ubuntu-x64 ({e}), chuyển sang dùng binary trong Dataset Note 1...")

if not LLAMA_SERVER_BIN:
    existing_server = find_existing_file("llama-server")
    if existing_server:
        src_bin_dir = os.path.dirname(existing_server)
        if os.path.exists(TMP_BIN_DIR):
            shutil.rmtree(TMP_BIN_DIR)
        shutil.copytree(src_bin_dir, TMP_BIN_DIR, symlinks=True)
        for cuda_so in glob.glob(os.path.join(TMP_BIN_DIR, "**", "*cuda*.so*"), recursive=True):
            try:
                os.remove(cuda_so)
            except Exception:
                pass
        fix_permissions_and_symlinks(TMP_BIN_DIR)
        cand = os.path.join(TMP_BIN_DIR, "llama-server")
        ok, out = test_cpu_binary(cand)
        if ok:
            LLAMA_SERVER_BIN = cand
            print(f"✅ Đã nạp llama-server từ Dataset Note 1 (chế độ CPU): {LLAMA_SERVER_BIN}")
        else:
            raise RuntimeError(f"Binary trong Dataset không chạy được trên CPU: {out}")
    else:
        raise RuntimeError("Không tìm thấy llama-server trong Dataset và không thể tải từ GitHub.")

# ==========================================
# 2. ĐỊNH VỊ & COPY MODEL 1 (AI CHÍNH 27B) SANG SSD CỤC BỘ
# ==========================================
MTP_FILENAME = "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0-MTP.gguf"
PLAIN_FILENAME = "Ternary-Bonsai-2-27B-Abliterated-v2-PQ2_0.gguf"

if PREFER_MTP_FILE:
    RAW_MODEL_PATH = find_existing_file(MTP_FILENAME) or find_existing_file(PLAIN_FILENAME)
    TARGET_MODEL_FILENAME = MTP_FILENAME
else:
    RAW_MODEL_PATH = find_existing_file(PLAIN_FILENAME) or find_existing_file(MTP_FILENAME)
    TARGET_MODEL_FILENAME = PLAIN_FILENAME

if RAW_MODEL_PATH:
    MODEL_FILENAME = os.path.basename(RAW_MODEL_PATH)
    print(f"Tìm thấy [Model 1 - AI] trong Dataset Note 1: {RAW_MODEL_PATH}")
    MODEL_PATH = stage_to_local_ssd(RAW_MODEL_PATH, "Model 1 - AI 27B")
else:
    MODEL_FILENAME = TARGET_MODEL_FILENAME
    print(f"⚠️ Không thấy {MODEL_FILENAME} trong /kaggle/input! Đang tải tự động về {LOCAL_SSD_DIR}...")
    MODEL_PATH = hf_hub_download(repo_id=REPO_ID, filename=MODEL_FILENAME, local_dir=LOCAL_SSD_DIR)
    print(f"✅ [Model 1 - AI] Đã tải xong: {MODEL_PATH}")

# ==========================================
# 3. ĐỊNH VỊ & COPY MODEL 2 (VISION MMPROJ) SANG SSD CỤC BỘ
# ==========================================
MMPROJ_PATH = None
MMPROJ_FILENAME = "Ternary-Bonsai-2-27B-mmproj-Q8_0.gguf"

if USE_VISION:
    RAW_MMPROJ_PATH = find_existing_file(MMPROJ_FILENAME) or find_existing_file("*mmproj*.gguf")
    if RAW_MMPROJ_PATH:
        print(f"Tìm thấy [Model 2 - Nhìn Ảnh Vision] trong Dataset Note 1: {RAW_MMPROJ_PATH}")
        MMPROJ_PATH = stage_to_local_ssd(RAW_MMPROJ_PATH, "Model 2 - Vision mmproj")
    else:
        print(f"⚠️ Không thấy {MMPROJ_FILENAME} trong Dataset! Đang tải bổ sung (0.63 GB) về {LOCAL_SSD_DIR}...")
        MMPROJ_PATH = hf_hub_download(repo_id=REPO_ID, filename=MMPROJ_FILENAME, local_dir=LOCAL_SSD_DIR)
        print(f"✅ [Model 2 - Nhìn Ảnh Vision] Đã tải xong: {MMPROJ_PATH}")

## 3. Khởi chạy `llama-server` (Chế độ CPU 30GB RAM) + Mở Ngrok + Vòng lặp Chạy Full Giờ Kaggle

In [ ]:
import datetime
import sys


def is_port_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def get_secret(name, fallback=""):
    val = os.environ.get(name, "").strip()
    if val:
        return val
    if PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
    elif PLATFORM == "colab":
        try:
            from google.colab import userdata
            return userdata.get(name).strip()
        except Exception:
            pass
    return fallback.strip()


def get_ram_usage_str():
    try:
        with open("/proc/meminfo", "r") as f:
            mem = f.read()
        mt = int(re.search(r"MemTotal:\s+(\d+)", mem).group(1)) / (1024 ** 2)
        ma = int(re.search(r"MemAvailable:\s+(\d+)", mem).group(1)) / (1024 ** 2)
        used = mt - ma
        return f"{used:.1f}/{mt:.1f}GB"
    except Exception:
        return "N/A"


def start_llama_server_cpu(port):
    subprocess.run(["pkill", "-f", "llama-server"], check=False)
    time.sleep(1)
    while not is_port_free(port):
        port += 1

    if DEFAULT_THINKING:
        reasoning_flags = ["--reasoning", "on", "--chat-template-kwargs", json.dumps({"reasoning_effort": REASONING_EFFORT})]
    else:
        reasoning_flags = ["--reasoning", "off", "--reasoning-budget", "0"]

    cmd = [
        LLAMA_SERVER_BIN,
        "-m", MODEL_PATH,
        "-ngl", "0",                     # Chạy 100% trên CPU
        "-t", str(CPU_THREADS),          # Số luồng CPU decode
        "-tb", str(CPU_THREADS),         # Số luồng CPU prefill/xử lý ảnh
        "-b", str(BATCH_SIZE),
        "-ub", str(UBATCH_SIZE),
        "-fa", "auto",
        "-c", str(CONTEXT_SIZE),
        "--parallel", "1",
        "--load-mode", LOAD_MODE,        # Dùng mmap trên SSD cục bộ để nạp tức thì
        "--no-warmup",                   # Bỏ qua bước chạy nháp 27B khi khởi động để mở cổng ngay lập tức
        "--cache-type-k", KV_CACHE_TYPE,
        "--cache-type-v", KV_CACHE_TYPE,
        "--jinja",
        "--temp", "1.0",
        "--top-k", "20",
        "--top-p", "0.95",
        "--metrics",
        "--host", "127.0.0.1",
        "--port", str(port),
    ] + reasoning_flags

    if USE_MTP_DRAFTING and "MTP" in os.path.basename(MODEL_PATH):
        cmd.extend(["--spec-type", "draft-mtp", "--spec-draft-n-max", "2"])

    if MMPROJ_PATH:
        cmd.extend([
            "--mmproj", MMPROJ_PATH,
            "--no-mmproj-offload",
            "--image-min-tokens", str(IMAGE_MIN_TOKENS),
            "--image-max-tokens", str(IMAGE_MAX_TOKENS),
        ])

    bin_dir = os.path.dirname(LLAMA_SERVER_BIN)
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = f"{bin_dir}:" + env.get("LD_LIBRARY_PATH", "")

    log_path = "/tmp/llama_server_cpu.log"
    proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, env=env)

    print(f"Đang nạp 2 Model (AI 27B + Vision mmproj) trên cổng {port} (theo dõi log trực tiếp)...")
    ready = False
    last_pos = 0
    for _ in range(180):
        # In các dòng log mới nhất để không bị đứng im lặng
        try:
            with open(log_path, "r", errors="replace") as lf:
                lf.seek(last_pos)
                new_txt = lf.read()
                last_pos = lf.tell()
            if new_txt.strip():
                for line in new_txt.splitlines():
                    if "unused tensor" not in line:
                        print(f"[llama-server] {line}")
        except Exception:
            pass

        if proc.poll() is not None:
            raise RuntimeError(f"llama-server thoát sớm với mã {proc.returncode}")
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                ready = True
                break
        except Exception:
            pass
        time.sleep(2)

    if not ready:
        raise TimeoutError("llama-server không phản hồi /health sau 360 giây.")
    return proc, port


def start_ngrok_tunnel(port, authtoken, domain):
    try:
        ngrok.kill()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "ngrok"], check=False)
    time.sleep(1)

    ngrok_path = conf.get_default().ngrok_path
    if not os.path.exists(ngrok_path):
        install_ngrok(ngrok_path)
    subprocess.run([ngrok_path, "config", "add-authtoken", authtoken], check=True)

    ngrok_cmd = [ngrok_path, "http", str(port), "--log=stdout"]
    if domain:
        ngrok_cmd.extend(["--domain", domain])
    proc = subprocess.Popen(ngrok_cmd, stdout=open("/tmp/ngrok.log", "w"), stderr=subprocess.STDOUT)

    public_url = None
    for _ in range(30):
        try:
            tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json().get("tunnels", [])
            https_tunnels = [t for t in tunnels if t.get("public_url", "").startswith("https://")]
            if https_tunnels:
                public_url = https_tunnels[0]["public_url"].rstrip("/")
                break
        except Exception:
            pass
        time.sleep(1)

    if not public_url:
        raise TimeoutError("Không lấy được PUBLIC_URL từ Ngrok.")
    return proc, public_url


def inject_browser_keepalive():
    try:
        from IPython.display import Javascript, display
        js_code = """
        (function() {
            if (window.__keepAliveInjected) return;
            window.__keepAliveInjected = true;
            setInterval(function() {
                document.querySelectorAll('mwc-button, paper-button, button').forEach(b => {
                    const txt = (b.innerText || b.textContent || '').trim().toLowerCase();
                    if (txt === 'ok' || txt === 'reconnect' || txt === 'keep working' || txt === 'stay connected') {
                        b.click();
                    }
                });
                document.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: Math.random()*100, clientY: Math.random()*100}));
            }, 30000);
            console.log('[Keep-Alive] Kaggle CPU anti-idle active.');
        })();
        """
        display(Javascript(js_code))
    except Exception:
        pass


NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", MANUAL_NGROK_AUTHTOKEN)
NGROK_DOMAIN = re.sub(r"^https?://", "", get_secret("NGROK_DOMAIN", MANUAL_NGROK_DOMAIN)).strip("/")
if not NGROK_AUTHTOKEN:
    raise ValueError("Thiếu NGROK_AUTHTOKEN! Hãy bật trong Kaggle Secrets (Add-ons -> Secrets) hoặc điền vào MANUAL_NGROK_AUTHTOKEN ở Cell 1.")

server_proc, ACTIVE_PORT = start_llama_server_cpu(SERVER_PORT)
ngrok_proc, PUBLIC_URL = start_ngrok_tunnel(ACTIVE_PORT, NGROK_AUTHTOKEN, NGROK_DOMAIN)

print("=" * 74)
print(f"NỀN TẢNG          : {PLATFORM.upper()} CPU ({CPU_THREADS} Threads | RAM đang dùng: {get_ram_usage_str()})")
print(f"WEB UI TRỰC TIẾP  : {PUBLIC_URL}")
print(f"OPENAI BASE URL   : {PUBLIC_URL}/v1")
print(f"MODEL 1 (AI)      : {MODEL_FILENAME}")
print(f"MODEL 2 (NHÌN ẢNH): {os.path.basename(MMPROJ_PATH) if MMPROJ_PATH else 'TẮT'} (Max {IMAGE_MAX_TOKENS} vision tokens/ảnh)")
print(f"CONTEXT / CACHE   : {CONTEXT_SIZE} ({KV_CACHE_TYPE}) | Thinking mặc định: {'BẬT' if DEFAULT_THINKING else 'TẮT'}")
print("=" * 74)

# ==========================================
# VÒNG LẶP DUY TRÌ KẾT NỐI FULL GIỜ KAGGLE (ĐẾN KHI CHẠM MỨC GIỜ 12H)
# ==========================================
if ENABLE_KEEP_ALIVE:
    if KEEP_ALIVE_ENABLE_BROWSER_JS:
        inject_browser_keepalive()

    target_seconds = int(KEEP_ALIVE_MAX_HOURS * 3600)
    start_ts = time.time()
    end_dt = datetime.datetime.now() + datetime.timedelta(seconds=target_seconds)

    if KEEP_ALIVE_UNTIL_LIMIT:
        print(f"\n[KEEP-ALIVE] Đang chạy chế độ FULL GIỜ liên tục cho đến khi Kaggle chạm giới hạn phiên (~12 giờ, dự kiến {end_dt.strftime('%H:%M:%S %d/%m')}).")
    else:
        print(f"\n[KEEP-ALIVE] Đang duy trì kết nối {KEEP_ALIVE_MAX_HOURS} giờ (đến {end_dt.strftime('%H:%M:%S %d/%m')}).")
    print("[KEEP-ALIVE] Lưu ý: Không gửi dummy prompt chiếm slot để bạn có thể chat & gửi ảnh bất cứ lúc nào mà không bị nghẽn hàng đợi.")
    print("[KEEP-ALIVE] Nhấn nút Stop (■) trên cell này nếu muốn dừng thủ công.\n")

    try:
        while True:
            now = time.time()
            elapsed = now - start_ts
            if (not KEEP_ALIVE_UNTIL_LIMIT) and (elapsed >= target_seconds):
                print(f"\n[KEEP-ALIVE] Đã đạt mốc {KEEP_ALIVE_MAX_HOURS} giờ!")
                break

            try:
                with open("/tmp/.kaggle_cpu_keepalive", "w") as hb:
                    hb.write(f"{now}\n")
            except Exception:
                pass

            server_alive = (server_proc.poll() is None)
            if server_alive:
                try:
                    r = requests.get(f"http://127.0.0.1:{ACTIVE_PORT}/health", timeout=5)
                    server_alive = (r.status_code in (200, 503))
                except Exception:
                    server_alive = False

            if not server_alive:
                print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] ⚠️ Phát hiện llama-server ngừng hoạt động!")
                if KEEP_ALIVE_AUTO_RESTART:
                    print("[KEEP-ALIVE] Đang tự động nạp lại 2 model...")
                    server_proc, ACTIVE_PORT = start_llama_server_cpu(ACTIVE_PORT)
                else:
                    break

            ngrok_alive = (ngrok_proc.poll() is None)
            if ngrok_alive:
                try:
                    tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=3).json().get("tunnels", [])
                    ngrok_alive = len(tunnels) > 0
                except Exception:
                    ngrok_alive = False

            if not ngrok_alive:
                print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] ⚠️ Phát hiện Ngrok bị ngắt!")
                if KEEP_ALIVE_AUTO_RESTART:
                    print("[KEEP-ALIVE] Đang mở lại Ngrok tunnel...")
                    ngrok_proc, PUBLIC_URL = start_ngrok_tunnel(ACTIVE_PORT, NGROK_AUTHTOKEN, NGROK_DOMAIN)
                    print(f"[KEEP-ALIVE] URL mới: {PUBLIC_URL}/v1")
                else:
                    break

            hrs_done = elapsed / 3600.0
            hrs_left = max(0.0, (target_seconds - elapsed) / 3600.0)
            status_line = (
                f"\r[{datetime.datetime.now().strftime('%H:%M:%S')}] "
                f"Đã chạy: {hrs_done:.2f}h / {KEEP_ALIVE_MAX_HOURS:.1f}h (còn ~{hrs_left:.2f}h) | "
                f"RAM: {get_ram_usage_str()} | Server: OK | Vision: {'ON' if MMPROJ_PATH else 'OFF'} | Ngrok: OK"
            )
            sys.stdout.write(status_line + "   ")
            sys.stdout.flush()

            time.sleep(KEEP_ALIVE_PING_INTERVAL)
    except KeyboardInterrupt:
        print("\n[KEEP-ALIVE] Đã dừng vòng lặp theo yêu cầu người dùng (Server & Ngrok vẫn chạy nền).")

## 4. Tắt Server & Ngrok sạch sẽ (Tùy chọn khi muốn dừng hẳn)

In [ ]:
def stop_all():
    try:
        ngrok.kill()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "ngrok"], check=False)
    subprocess.run(["pkill", "-f", "llama-server"], check=False)
    print("Đã tắt Ngrok và llama-server sạch sẽ.")

# Bỏ dấu # ở dòng dưới và chạy cell này khi muốn tắt hẳn server:
# stop_all()